# TRIBE v2 — Brain Mapping Inference on Your Videos

**Runtime → Change runtime type → T4 GPU** before running.

Upload your videos to the `posted/` folder after running the setup cell.

In [ ]:
# Step 1: Install TRIBE v2
# Pin numpy<2 first — some tribev2 deps (neuralset/nilearn) are compiled against numpy 1.x
# and will fail with "cannot import _center from numpy_core.umath" on numpy 2.x
!pip install -q "numpy<2"
!pip install -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

In [ ]:
# Step 2: Login to HuggingFace (needed for LLaMA 3.2)
# Make sure you've accepted the license at https://huggingface.co/meta-llama/Llama-3.2-3B
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# Step 3: Upload videos
import os
os.makedirs('posted', exist_ok=True)

from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'posted/{name}', 'wb') as f:
        f.write(data)
    print(f'Saved: posted/{name}')

In [ ]:
# Step 4: Load model
from pathlib import Path
from tribev2.demo_utils import TribeModel
from tribev2.plotting import PlotBrain

CACHE = Path('./cache')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE)
plotter = PlotBrain(mesh='fsaverage5')
print('Model loaded on', next(model._model.parameters()).device)

In [ ]:
# Step 5: Run inference on all videos
import json
import numpy as np

VIDEO_DIR = Path('posted')
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)

videos = sorted(VIDEO_DIR.glob('*.mp4'))
print(f'Found {len(videos)} videos:\n')
for v in videos:
    print(f'  - {v.name}')

results_summary = {}

for i, video_path in enumerate(videos):
    print(f'\n{"="*60}')
    print(f'[{i+1}/{len(videos)}] Processing: {video_path.name}')
    print(f'{"="*60}')

    try:
        df = model.get_events_dataframe(video_path=str(video_path))
        preds, segments = model.predict(events=df)
        print(f'  Predictions shape: {preds.shape}')

        safe_name = video_path.stem.replace(' ', '_')
        np.save(OUTPUT_DIR / f'{safe_name}_preds.npy', preds)

        n_timesteps, n_vertices = preds.shape
        n_hemi = n_vertices // 2
        lh_preds = preds[:, :n_hemi]
        rh_preds = preds[:, n_hemi:]

        summary = {
            'video': video_path.name,
            'n_timesteps': int(n_timesteps),
            'n_vertices': int(n_vertices),
            'duration_seconds': int(n_timesteps),
            'overall_mean_activation': float(np.mean(preds)),
            'overall_std_activation': float(np.std(preds)),
            'overall_max_activation': float(np.max(preds)),
            'left_hemisphere_mean': float(np.mean(lh_preds)),
            'right_hemisphere_mean': float(np.mean(rh_preds)),
            'left_hemisphere_max': float(np.max(lh_preds)),
            'right_hemisphere_max': float(np.max(rh_preds)),
            'peak_timestep': int(np.argmax(np.mean(preds, axis=1))),
        }

        results_summary[video_path.name] = summary
        print(f'  Mean activation: {summary["overall_mean_activation"]:.4f}')
        print(f'  Max activation: {summary["overall_max_activation"]:.4f}')
        print(f'  LH mean: {summary["left_hemisphere_mean"]:.4f} | RH mean: {summary["right_hemisphere_mean"]:.4f}')
        print(f'  Peak activity at timestep: {summary["peak_timestep"]}s')

    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()
        results_summary[video_path.name] = {'error': str(e)}

with open(OUTPUT_DIR / 'brain_mapping_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f'\n{"="*60}')
print('SUMMARY')
print(f'{"="*60}')
for name, res in results_summary.items():
    if 'error' in res:
        print(f'  {name}: FAILED - {res["error"]}')
    else:
        print(f'  {name}: {res["n_timesteps"]}s, mean={res["overall_mean_activation"]:.4f}, max={res["overall_max_activation"]:.4f}')

In [ ]:
# Step 6: Visualize brain mappings for each video
for name, res in results_summary.items():
    if 'error' in res:
        continue
    safe_name = Path(name).stem.replace(' ', '_')
    preds = np.load(OUTPUT_DIR / f'{safe_name}_preds.npy')
    # Load segments for this video
    df = model.get_events_dataframe(video_path=str(VIDEO_DIR / name))
    _, segments = model.predict(events=df)

    print(f'\n--- {name} ---')
    n_show = min(15, preds.shape[0])
    fig = plotter.plot_timesteps(
        preds[:n_show],
        segments=segments[:n_show],
        cmap='fire',
        norm_percentile=99,
        vmin=0.5,
        alpha_cmap=(0, 0.2),
        show_stimuli=True,
    )
    fig.show()

In [ ]:
# Step 7: Download results
!zip -r results.zip results/
files.download('results.zip')